In [2]:
import ee
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai_geometry = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

# Fetch historical NO₂ data for decomposition
def fetch_no2_historical_data(end_date, duration_months=24):
    """
    Fetch historical NO₂ data over a specified duration for decomposition.
    """
    start_date = ee.Date(end_date).advance(-duration_months, 'month')
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
        .select('tropospheric_NO2_column_number_density') \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai_geometry)

    # Aggregate NO₂ values into a time series (e.g., monthly means)
    def reduce_to_time_series(img):
        mean = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=mumbai_geometry,
            scale=10,
            maxPixels=1e9
        )
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'value': mean.get('tropospheric_NO2_column_number_density')
        })

    time_series = collection.map(reduce_to_time_series).getInfo()
    dates = [entry['properties']['date'] for entry in time_series['features']]
    values = [entry['properties']['value'] for entry in time_series['features'] if entry['properties']['value'] is not None]

    return np.array(dates), np.array(values)

# Apply decomposition to predict NO₂
def apply_decomposition(target, model_type='additive'):
    """
    Decompose the time series and reconstruct NO₂ values.
    """
    print("Decomposing time series...")
    decomposition = seasonal_decompose(target, model=model_type, period=12, extrapolate_trend='freq')

    # Combine decomposed components for prediction
    trend = decomposition.trend
    seasonal = decomposition.seasonal
    residual = decomposition.resid

    print("Decomposition complete. Reconstructing NO₂...")
    reconstructed_no2 = trend + seasonal + residual
    return reconstructed_no2

# Fetch and predict NO₂ for a specific date
def fetch_and_predict(user_date):
    """
    Fetch historical data, apply decomposition, and prepare NO₂ layer for export.
    """
    print(f"Fetching data for {user_date}...")

    # Fetch historical NO₂ data for decomposition
    dates, values = fetch_no2_historical_data(user_date)

    if len(values) < 24:
        raise ValueError(f"Insufficient data for decomposition. Only {len(values)} observations found.")

    print("Preparing NO₂ data for decomposition...")
    reconstructed_no2 = apply_decomposition(values)

    # Use the most recent NO₂ prediction as the downscaled value
    predicted_no2_value = reconstructed_no2[-1]

    # Generate the NO₂ image for export
    no2_image = ee.Image.constant(predicted_no2_value).clip(mumbai_geometry).rename('Downscaled_NO2')

    return no2_image

# Export the downscaled NO₂ layer
def export_to_gee(image, user_date):
    """
    Export the predicted NO₂ layer to Google Earth Engine.
    """
    print(f"Exporting downscaled NO₂ layer for {user_date}...")
    task = ee.batch.Export.image.toAsset(
        image=image,
        description=f'Downscaled_NO2_{user_date}',
        assetId=f'projects/ee-chandanagalgali/assets/downscaled_NO2_{user_date}',
        region=mumbai_geometry,
        scale=10,
        maxPixels=1e13
    )
    task.start()
    print(f"Export task started for {user_date}. Monitor progress in GEE Tasks.")

if __name__ == "__main__":
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    try:
        print(f"Processing downscaled NO₂ layer for {user_date}...")
        downscaled_image = fetch_and_predict(user_date)
        export_to_gee(downscaled_image, user_date)
        print("Processing complete. Task submitted to GEE.")
    except Exception as e:
        print(f"Error: {e}")

Enter the date for prediction (YYYY-MM-DD): 2023-12-10
Processing downscaled NO₂ layer for 2023-12-10...
Fetching data for 2023-12-10...
Error: Computation timed out.
